# RosettaBench — Stratified Dataset Builder

Builds a **150-problem stratified sample** (40 Easy, 50 Medium, 60 Hard) from LiveCodeBench for use with RosettaBench.

In [1]:
%pip install datasets -q

Note: you may need to restart the kernel to use updated packages.


## 1. Load LiveCodeBench (release_v5, STDIN-only)

In [2]:
import json
import pyarrow.parquet as pq
import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download

REPO_ID = "sam-paech/livecodebench-code_generation_lite"

# Fetch only the release_v5 shards
all_files = list(list_repo_files(REPO_ID, repo_type="dataset"))
v5_files = sorted(f for f in all_files if f.startswith("data/release_v5") and f.endswith(".parquet"))
print(f"Downloading {len(v5_files)} release_v5 shard(s)...")

shards = []
for fname in v5_files:
    local_path = hf_hub_download(repo_id=REPO_ID, filename=fname, repo_type="dataset")
    shards.append(pq.read_table(local_path).to_pandas())

lcb_df = pd.concat(shards, ignore_index=True)
print(f"Total rows loaded: {len(lcb_df)}")

# Filter to STDIN/STDOUT problems (AtCoder only for platform consistency)
stdin_mask = lcb_df["platform"] == "atcoder"
lcb_stdin = lcb_df[stdin_mask].copy()

def _safe_json(val):
    if val is None:
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, bytes):
        val = val.decode("utf-8", errors="replace")
    if not isinstance(val, str) or not val.strip():
        return []
    try:
        result = json.loads(val)
        return result if isinstance(result, list) else []
    except json.JSONDecodeError:
        return []

def parse_tests(row):
    public  = _safe_json(row.get("public_test_cases"))
    private = _safe_json(row.get("private_test_cases"))
    return public + private

lcb_stdin["all_tests"] = lcb_stdin.apply(parse_tests, axis=1)
lcb_stdin = lcb_stdin[lcb_stdin["all_tests"].apply(len) > 0].reset_index(drop=True)

# Keep only problems with at least MIN_TEST_CASES for robust verification
MIN_TEST_CASES = 3
lcb_stdin = lcb_stdin[lcb_stdin["all_tests"].apply(len) >= MIN_TEST_CASES].reset_index(drop=True)
print(f"Problems with >= {MIN_TEST_CASES} test cases: {len(lcb_stdin)}")

print(f"\nSTDIN problems with tests: {len(lcb_stdin)}")
print(f"\nDifficulty distribution (full pool):")
print(lcb_stdin["difficulty"].value_counts().to_string())

data/release_v5-00000-of-00009.parquet:   0%|          | 0.00/65.2M [00:00<?, ?B/s]

data/release_v5-00001-of-00009.parquet:   0%|          | 0.00/47.0M [00:00<?, ?B/s]

data/release_v5-00002-of-00009.parquet:   0%|          | 0.00/591M [00:00<?, ?B/s]

data/release_v5-00003-of-00009.parquet:   0%|          | 0.00/427M [00:00<?, ?B/s]

data/release_v5-00004-of-00009.parquet:   0%|          | 0.00/750M [00:00<?, ?B/s]

data/release_v5-00005-of-00009.parquet:   0%|          | 0.00/612M [00:00<?, ?B/s]

data/release_v5-00006-of-00009.parquet:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

data/release_v5-00007-of-00009.parquet:   0%|          | 0.00/492M [00:00<?, ?B/s]

data/release_v5-00008-of-00009.parquet:   0%|          | 0.00/93.7M [00:00<?, ?B/s]

Total rows loaded: 880
Problems with >= 3 test cases: 342

STDIN problems with tests: 342

Difficulty distribution (full pool):
difficulty
easy      117
medium    113
hard      112


## 2. Stratified Sampling: 40 Easy + 50 Medium + 60 Hard

In [3]:
# ============================================================
# Stratified sampling configuration
# ============================================================

RANDOM_SEED = 42              # Reproducible sampling

# Validate difficulty column values
expected_difficulties = {"easy", "medium", "hard"}
actual_difficulties = set(lcb_stdin["difficulty"].str.lower().unique())
unexpected = actual_difficulties - expected_difficulties
if unexpected:
    print(f"⚠ WARNING: unexpected difficulty values found: {unexpected}")

difficulty_targets = {
    "easy":   40,
    "medium": 50,
    "hard":   60,
}

strata = []
for diff, n in difficulty_targets.items():
    pool = lcb_stdin[lcb_stdin["difficulty"].str.lower() == diff]
    available = len(pool)
    take = min(n, available)
    sampled = pool.sample(n=take, random_state=RANDOM_SEED)
    strata.append(sampled)
    print(f"{diff.capitalize():>6}: sampled {take} / {available} available")

sample_df = pd.concat(strata, ignore_index=True)
print(f"\nTotal problems selected: {len(sample_df)}")


  Easy: sampled 40 / 117 available
Medium: sampled 50 / 113 available
  Hard: sampled 60 / 112 available

Total problems selected: 150


In [4]:
# ============================================================
# Validation checks
# ============================================================

assert len(sample_df) == 150, f"Expected 150 problems, got {len(sample_df)}"

for diff, expected_n in {"easy": 40, "medium": 50, "hard": 60}.items():
    count = (sample_df["difficulty"].str.lower() == diff).sum()
    assert count == expected_n, f"Expected {expected_n} {diff} problems, got {count}"

assert sample_df["question_id"].is_unique, "Duplicate question_ids found!"

min_tests = sample_df["all_tests"].apply(len).min()
assert min_tests >= 1, f"Found problem(s) with 0 test cases"

print("✓ All validation checks passed")


✓ All validation checks passed


## 3. Dataset Summary

In [5]:
print("=" * 60)
print("STRATIFIED SAMPLE SUMMARY")
print("=" * 60)
print(f"\nTotal problems: {len(sample_df)}")
print(f"\nBy difficulty:")
print(sample_df["difficulty"].value_counts().to_string())
print(f"\nBy platform:")
print(sample_df["platform"].value_counts().to_string())
print(f"\nAvg test cases per problem: {sample_df['all_tests'].apply(len).mean():.1f}")
print(f"Min test cases: {sample_df['all_tests'].apply(len).min()}")
print(f"Max test cases: {sample_df['all_tests'].apply(len).max()}")

# Show a few problem IDs per difficulty
for diff in ["easy", "medium", "hard"]:
    subset = sample_df[sample_df["difficulty"].str.lower() == diff]
    ids = subset["question_id"].head(5).tolist()
    print(f"\n{diff.capitalize()} examples: {ids}")

STRATIFIED SAMPLE SUMMARY

Total problems: 150

By difficulty:
difficulty
hard      60
medium    50
easy      40

By platform:
platform
atcoder    150

Avg test cases per problem: 3.2
Min test cases: 3
Max test cases: 6

Easy examples: ['abc333_b', 'abc304_b', 'abc342_a', 'abc331_b', 'abc309_b']

Medium examples: ['abc363_c', 'abc302_d', 'abc330_c', 'abc355_c', 'abc307_c']

Hard examples: ['abc345_d', 'abc367_g', 'abc305_e', 'abc352_d', 'abc348_d']


## 4. Export for Benchmark Runs

In [6]:
# Save to parquet for reuse across runs
OUTPUT_PATH = "rosetta_150_stratified-compressed.parquet"
sample_df.to_parquet(OUTPUT_PATH, index=False, compression="zstd")
print(f"Saved {len(sample_df)} problems to {OUTPUT_PATH}")
print(f"Columns: {list(sample_df.columns)}")


Saved 150 problems to rosetta_150_stratified-compressed.parquet
Columns: ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata', 'all_tests']
